# 🧬 Project Prometheus - FreeCiv Curriculum Training (Colab)

## Multi-Day Strategic AI Training on Google Colab

This notebook trains Prometheus agents to play FreeCiv through a 4-stage curriculum:
- **Stage 1**: Novice AI (learn basics)
- **Stage 2**: Easy AI (develop tactics)
- **Stage 3**: Normal AI (master strategy)
- **Stage 4**: Hard AI (expert play)

**⚠️ WARNING**: Full curriculum can take **8+ days**. Use checkpoints!

**Strategy for Colab**:
- Run one stage per session (~12 hours)
- Save checkpoints to Google Drive
- Resume from checkpoint in next session

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/Prometheus_v0_PoC/blob/main/Prometheus_FreeCiv_Colab.ipynb)

## 1. Setup Colab Environment

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Mount Google Drive for checkpoints
from google.colab import drive
drive.mount('/content/drive')

# Create checkpoint directory
!mkdir -p /content/drive/MyDrive/Prometheus_FreeCiv_Checkpoints

print("✓ Google Drive mounted")
print("✓ Checkpoint directory ready")

## 2. Install Dependencies

In [ ]:
# Clone Prometheus repository
!git clone https://github.com/YOUR_USERNAME/Prometheus_v0_PoC.git
%cd Prometheus_v0_PoC

# Install requirements
!pip install -q -r requirements.txt

# Install FreeCiv (if not using API)
!apt-get update -qq
!apt-get install -y -qq freeciv-server freeciv-client-gtk3

print("✓ All dependencies installed")

In [ ]:
# Configure API keys
import os
from google.colab import userdata

# Get API key from Colab secrets (recommended)
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY
    print("✓ API key loaded from Colab secrets")
except:
    print("⚠️  No API key found in secrets. Will use free tier.")
    GOOGLE_API_KEY = None

## 3. Import Prometheus Components

In [ ]:
import sys
import logging
import json
from datetime import datetime
from pathlib import Path

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s - %(message)s'
)

# Import Prometheus
from prometheus.generalist_planner import GeneralistPlannerAgent
from prometheus.smm import EvolutionaryOrchestratorAgent, EvolutionConfig
from prometheus.iee import IntrospectionEvaluationEngine
from prometheus.domain_expert_agent import GamePlayingExpertAgent
from benchmarks.prometheus_bench_v0_2 import PrometheusBenchV02
from benchmarks.freeciv_strategic_opponents import (
    FREECIV_CURRICULUM, get_opponent_config, estimate_training_time
)

# Visualization
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

print("✓ All components imported")

## 4. Define FreeCiv Curriculum

4-stage progressive difficulty training:

In [ ]:
# FreeCiv curriculum (adapted for Colab runtime limits)
CURRICULUM = [
    {
        "stage": 1,
        "name": "Baseline Training",
        "opponent": "novice",
        "description": "Learn basic mechanics vs Novice AI",
        "population": 4,        # Smaller for Colab
        "generations": 5,       # Fewer for 12-hour limit
        "target_fitness": 0.30,
        "mutation_rate": 0.7,
        "elitism": 1,
    },
    {
        "stage": 2,
        "name": "Tactical Development",
        "opponent": "easy",
        "description": "Develop tactical skills vs Easy AI",
        "population": 6,
        "generations": 8,
        "target_fitness": 0.20,
        "mutation_rate": 0.6,
        "elitism": 2,
    },
    {
        "stage": 3,
        "name": "Strategic Training",
        "opponent": "normal",
        "description": "Master strategy vs Normal AI",
        "population": 8,
        "generations": 10,
        "target_fitness": 0.15,
        "mutation_rate": 0.5,
        "elitism": 2,
    },
    {
        "stage": 4,
        "name": "Expert Training",
        "opponent": "hard",
        "description": "Expert play vs Hard AI",
        "population": 10,
        "generations": 12,
        "target_fitness": 0.10,
        "mutation_rate": 0.4,
        "elitism": 3,
    },
]

print("📚 FREECIV CURRICULUM (Colab-Optimized):")
print("─" * 70)
for stage in CURRICULUM:
    time_est = estimate_training_time(
        population_size=stage['population'],
        generations=stage['generations'],
        games_per_eval=3,  # Reduced for Colab
        minutes_per_game=20  # Faster game settings
    )
    print(f"Stage {stage['stage']}: {stage['name']}")
    print(f"  Opponent: {stage['opponent']}")
    print(f"  Target: {stage['target_fitness']:.0%} win rate")
    print(f"  Training: {stage['population']} agents × {stage['generations']} gens")
    print(f"  Estimated time: {time_est['total_hours']:.1f} hours")
    print()

total_time = sum([
    estimate_training_time(
        s['population'], s['generations'], 3, 20
    )['total_hours'] for s in CURRICULUM
])
print(f"📅 Total curriculum time: {total_time:.1f} hours ({total_time/24:.1f} days)")
print(f"\n💡 Strategy: Run {len(CURRICULUM)} separate Colab sessions, one per stage")

## 5. Checkpoint Management

Save/load progress to Google Drive:

In [ ]:
CHECKPOINT_DIR = Path("/content/drive/MyDrive/Prometheus_FreeCiv_Checkpoints")

def save_checkpoint(stage_num, stage_result, curriculum_results, all_fitness_history):
    """Save checkpoint to Google Drive"""
    checkpoint = {
        'stage': stage_num,
        'timestamp': datetime.now().isoformat(),
        'stage_result': stage_result,
        'curriculum_results': curriculum_results,
        'all_fitness_history': all_fitness_history,
    }
    
    checkpoint_file = CHECKPOINT_DIR / f"stage_{stage_num}_checkpoint.json"
    with open(checkpoint_file, 'w') as f:
        json.dump(checkpoint, f, indent=2)
    
    print(f"💾 Checkpoint saved: {checkpoint_file}")
    return checkpoint_file

def load_checkpoint(stage_num):
    """Load checkpoint from Google Drive"""
    checkpoint_file = CHECKPOINT_DIR / f"stage_{stage_num}_checkpoint.json"
    
    if checkpoint_file.exists():
        with open(checkpoint_file, 'r') as f:
            checkpoint = json.load(f)
        print(f"📂 Checkpoint loaded: {checkpoint_file}")
        return checkpoint
    else:
        print(f"⚠️  No checkpoint found for stage {stage_num}")
        return None

def list_checkpoints():
    """List all available checkpoints"""
    checkpoints = sorted(CHECKPOINT_DIR.glob("stage_*_checkpoint.json"))
    print("\n📂 Available Checkpoints:")
    for cp in checkpoints:
        print(f"  {cp.name}")
    return checkpoints

print("✓ Checkpoint system ready")

## 6. Select Which Stage to Run

**IMPORTANT**: Modify this cell to select which curriculum stage to train

In [ ]:
# ========================================
# CONFIGURE WHICH STAGE TO RUN
# ========================================

# Change this to run different stages:
STAGE_TO_RUN = 1  # 1, 2, 3, or 4

# Check for previous checkpoints
list_checkpoints()

# Load previous results if resuming
curriculum_results = []
all_fitness_history = []

if STAGE_TO_RUN > 1:
    print(f"\n🔄 Loading previous stage results...")
    for prev_stage in range(1, STAGE_TO_RUN):
        cp = load_checkpoint(prev_stage)
        if cp:
            curriculum_results.append(cp['stage_result'])
            all_fitness_history.extend(cp['all_fitness_history'])
    print(f"✓ Loaded {len(curriculum_results)} previous stages")

stage = CURRICULUM[STAGE_TO_RUN - 1]
print(f"\n🎯 Ready to run Stage {stage['stage']}: {stage['name']}")
print(f"  Opponent: {stage['opponent']}")
print(f"  Target: {stage['target_fitness']:.0%}")

## 7. Initialize Components

In [ ]:
# Initialize GeneralistPlanner
print("✓ Initializing GeneralistPlanner...")
planner = GeneralistPlannerAgent()

# Initialize components
print("✓ Initializing benchmark suite...")
benchmark_suite = PrometheusBenchV02()

print("✓ Initializing IEE...")
iee = IntrospectionEvaluationEngine(benchmark_suite=benchmark_suite)

print("\n✓ All components ready!")

## 8. Run Training for Selected Stage

**This will run for several hours! Colab may disconnect - use checkpoints to resume**

In [ ]:
print("="*70)
print(f"🎓 STAGE {stage['stage']}: {stage['name']}")
print("="*70)
print(f"\nOpponent: {stage['opponent']}")
print(f"Target: {stage['target_fitness']:.0%}")
print(f"Config: {stage['population']} agents × {stage['generations']} gens\n")

# Create config
config = EvolutionConfig(
    population_size=stage['population'],
    generations=stage['generations'],
    mutation_rate=stage['mutation_rate'],
    elitism_count=stage['elitism'],
    tournament_size=3,
    convergence_threshold=stage['target_fitness'],
    stagnation_generations=max(3, stage['generations'] // 4)
)

# Create SMM (use Gemini on Colab)
smm = EvolutionaryOrchestratorAgent(
    api_key=GOOGLE_API_KEY,
    config=config,
    prefer_local=False,  # Use Gemini API on Colab
)

print("🚀 Starting evolution...\n")
print("⚠️  This will take several hours!")
print("💡 If Colab disconnects, re-run and checkpoint will resume\n")

try:
    # Run evolution
    best_agent, fitness_history = smm.run_evolution(
        iee_evaluator=iee,
        benchmark_name="FreeCiv",
        template_class=GamePlayingExpertAgent
    )
    
    # Store results
    stage_result = {
        'stage': stage['stage'],
        'name': stage['name'],
        'opponent': stage['opponent'],
        'best_fitness': smm.best_fitness,
        'target_fitness': stage['target_fitness'],
        'achieved': smm.best_fitness >= stage['target_fitness'],
        'fitness_history': fitness_history
    }
    
    # Update cumulative results
    curriculum_results.append(stage_result)
    all_fitness_history.extend([{**h, 'stage': stage['stage']} for h in fitness_history])
    
    # Save checkpoint
    save_checkpoint(stage['stage'], stage_result, curriculum_results, all_fitness_history)
    
    print(f"\n✅ STAGE {stage['stage']} COMPLETE")
    print(f"Final: {smm.best_fitness:.1%} | Target: {stage['target_fitness']:.1%}")
    print(f"Status: {'✅ ACHIEVED' if stage_result['achieved'] else '⏸️ PARTIAL'}")
    
except KeyboardInterrupt:
    print("\n⚠️  Training interrupted - saving checkpoint...")
    save_checkpoint(stage['stage'], stage_result, curriculum_results, all_fitness_history)
    
except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback
    traceback.print_exc()

## 9. Visualize Results

View progress for completed stages:

In [ ]:
if curriculum_results:
    print("\n📊 CURRICULUM PROGRESS:")
    print("─"*70)
    print(f"{'Stage':<8} {'Opponent':<15} {'Final':<12} {'Target':<12} {'Status':<15}")
    print("─"*70)
    
    for result in curriculum_results:
        status = "✅ ACHIEVED" if result['achieved'] else "⏸️ PARTIAL"
        print(f"{result['stage']:<8} {result['opponent']:<15} "
              f"{result['best_fitness']:<12.1%} {result['target_fitness']:<12.1%} {status:<15}")
    print("─"*70)
    
    # Visualization
    if all_fitness_history:
        plt.figure(figsize=(12, 6))
        
        stage_colors = ['green', 'blue', 'purple', 'red']
        
        for stage_num in range(1, len(curriculum_results) + 1):
            stage_data = [h for h in all_fitness_history if h['stage'] == stage_num]
            if stage_data:
                generations = list(range(len(stage_data)))
                best_fitness = [h['best'] for h in stage_data]
                
                color = stage_colors[stage_num - 1] if stage_num <= len(stage_colors) else 'gray'
                label = CURRICULUM[stage_num - 1]['name']
                
                plt.plot(generations, best_fitness, f'{color}-o',
                        label=f'Stage {stage_num}: {label}', linewidth=2, markersize=4)
        
        plt.xlabel('Generation (within stage)', fontsize=11)
        plt.ylabel('Fitness (Win Rate)', fontsize=11)
        plt.title('FreeCiv Curriculum Learning Progress', fontsize=13, fontweight='bold')
        plt.legend(fontsize=9, loc='best')
        plt.grid(True, alpha=0.3)
        plt.ylim(-0.05, 0.5)
        
        plt.tight_layout()
        output_file = CHECKPOINT_DIR / 'freeciv_curriculum_progress.png'
        plt.savefig(output_file, dpi=150, bbox_inches='tight')
        print(f"\n✓ Visualization saved to: {output_file}")
        plt.show()
else:
    print("⚠️  No results to visualize yet")

## 10. Next Steps

**After this stage completes**:

1. **Save everything** (checkpoints are auto-saved to Google Drive)
2. **Start new Colab session** for next stage
3. **Change `STAGE_TO_RUN`** to the next number
4. **Re-run cells** - previous results will be loaded from checkpoints

**To resume if disconnected**:
- Just re-run all cells
- Checkpoints will automatically restore progress

**Curriculum Plan**:
- Session 1: Stage 1 (Novice AI) ✓
- Session 2: Stage 2 (Easy AI)
- Session 3: Stage 3 (Normal AI)
- Session 4: Stage 4 (Hard AI)

---

**🎉 Congratulations on completing a stage!**

Check your Google Drive for checkpoints and continue training in the next session!